In [ ]:
!pip install transformers datasets accelerate -q

In [ ]:
import numpy as np
import pandas as pd
import re
import unicodedata
import glob
import os
import torch

from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

# BERT

In [ ]:
all_user_df = pd.read_csv('receipt/train/all_user_6_label.csv')

## BERT 輸入：品名 [SEP] 店名 [SEP] 消費區間

In [ ]:
all_user_df["text"] = (
    all_user_df["item_clean"].astype(str)
    + " [SEP] "
    + all_user_df["store_clean"].astype(str)
    + " [SEP] "
    + all_user_df["price_bucket"].astype(str)
)

In [ ]:
all_user_df["group"] = (
    all_user_df["store_clean"]
    + "_"
    + all_user_df["item_clean"]
)

In [ ]:
all_user_df

,發票日期,消費明細_數量,消費明細_單價,消費明細_金額,store_clean,item_clean,label,user_id,price_bucket,text,group,label_id
0,2025-09-01,1.0,115.0,115.0,福康事業,法式特餐,飲食,0,LOW,法式特餐 [SEP] 福康事業 [SEP] LOW,福康事業_法式特餐,0
1,2025-09-01,1.0,95.0,95.0,一之軒食品南西分店,生吐司,飲食,0,LOW,生吐司 [SEP] 一之軒食品南西分店 [SEP] LOW,一之軒食品南西分店_生吐司,0
2,2025-09-02,1.0,990.0,990.0,新光三越百貨台北信義,咖啡廳coffee shops & tea salons,飲食,0,HIGH,咖啡廳coffee shops & tea salons [SEP] 新光三越百貨台北信義 ...,新光三越百貨台北信義_咖啡廳coffee shops & tea salons,0
3,2025-09-02,1.0,80.0,80.0,新光三越百貨台北信義,小吃food court,飲食,0,LOW,小吃food court [SEP] 新光三越百貨台北信義 [SEP] LOW,新光三越百貨台北信義_小吃food court,0
4,2025-09-03,1.0,35.0,35.0,統一超商台北市第148,桂格100%燕麥(顆粒微甜)290ml,飲食,0,VERY_LOW,桂格100%燕麥(顆粒微甜)290ml [SEP] 統一超商台北市第148 [SEP] VE...,統一超商台北市第148_桂格100%燕麥(顆粒微甜)290ml,0
...,...,...,...,...,...,...,...,...,...,...,...,...
2608,2026-05-21,1.0,39.0,39.0,全家便利商店台大二活門市部,香蕉可可三明治,飲食,2,VERY_LOW,香蕉可可三明治 [SEP] 全家便利商店台大二活門市部 [SEP] VERY_LOW,全家便利商店台大二活門市部_香蕉可可三明治,0
2609,2026-05-21,1.0,38.0,38.0,全家便利商店新北市第一一二,午后時光重乳奶茶,飲食,2,VERY_LOW,午后時光重乳奶茶 [SEP] 全家便利商店新北市第一一二 [SEP] VERY_LOW,全家便利商店新北市第一一二_午后時光重乳奶茶,0
2610,2026-05-22,1.0,65.0,65.0,和德昌台中學士路,雙倍or冰炫風,飲食,2,LOW,雙倍or冰炫風 [SEP] 和德昌台中學士路 [SEP] LOW,和德昌台中學士路_雙倍or冰炫風,0
2611,2026-05-23,1.0,330.0,330.0,勤美台中,星球工坊爆米花,飲食,2,MID,星球工坊爆米花 [SEP] 勤美台中 [SEP] MID,勤美台中_星球工坊爆米花,0


## split training & testing data

*   training: 80%
*   testing: 20%



In [ ]:
gss = GroupShuffleSplit(
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(all_user_df, groups=all_user_df["group"])
)

train_df = all_user_df.iloc[train_idx]
test_df = all_user_df.iloc[test_idx]

## Start training

In [ ]:
model_name = "ckiplab/bert-base-chinese"

tokenizer = AutoTokenizer.from_pretrained(model_name)

label2id = {
    "飲食": 0,
    "交通": 1,
    "購物": 2,
    "娛樂": 3,
    "教育": 4,
    "醫療健康": 5,
}

In [ ]:
train_encodings = tokenizer(
    train_df["text"].tolist(),
    truncation=True,
    padding=True,
    max_length=64
)


test_encodings = tokenizer(
    test_df["text"].tolist(),
    truncation=True,
    padding=True,
    max_length=64
)

In [ ]:
class ReceiptDataset(torch.utils.data.Dataset):

    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):

        item = {
            key: torch.tensor(val[idx])
            for key, val in self.encodings.items()
        }

        item["labels"] = torch.tensor(self.labels[idx])

        return item

    def __len__(self):
        return len(self.labels)

In [ ]:
train_dataset = ReceiptDataset(
    train_encodings,
    train_df["label_id"].tolist()
)


test_dataset = ReceiptDataset(
    test_encodings,
    test_df["label_id"].tolist()
)

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id,
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: ckiplab/bert-base-chinese
Key                                        | Status     | 
-------------------------------------------+------------+-
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
bert.pooler.dense.bias                     | MISSING    | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 
bert.pooler.dense.weight                   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you ex

In [ ]:
def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy_score(labels, predictions),

        "macro_f1": f1_score(
            labels,
            predictions,
            average="macro"
        ),

        "weighted_f1": f1_score(
            labels,
            predictions,
            average="weighted"
        )
    }

In [ ]:
training_args = TrainingArguments(

    output_dir="./results",

    learning_rate=2e-5,

    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,

    num_train_epochs=5,

    weight_decay=0.01,

    eval_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,

    report_to="none"
)

In [ ]:
trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=test_dataset,

    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1
1,No log,0.167113,0.958242,0.728610,0.953741
2,No log,0.170718,0.962637,0.865826,0.960898
3,No log,0.139304,0.975824,0.921033,0.975444
4,0.111693,0.169216,0.969231,0.884465,0.968932
5,0.111693,0.167190,0.971429,0.890808,0.971306


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=675, training_loss=0.08987650624027958, metrics={'train_runtime': 195.8191, 'train_samples_per_second': 55.102, 'train_steps_per_second': 3.447, 'total_flos': 354883780892160.0, 'train_loss': 0.08987650624027958, 'epoch': 5.0})

## Result

In [ ]:
pred_output = trainer.predict(test_dataset)

In [ ]:
preds = np.argmax(pred_output.predictions, axis=1)

labels = test_df["label_id"].values

In [ ]:
print(classification_report(
    labels,
    preds,
    target_names=list(label2id.keys())
))

              precision    recall  f1-score   support

          飲食       0.98      0.99      0.99       386
          交通       1.00      1.00      1.00         9
          購物       0.84      0.87      0.86        31
          娛樂       1.00      0.75      0.86         8
          教育       1.00      0.88      0.93        16
        醫療健康       1.00      0.80      0.89         5

    accuracy                           0.98       455
   macro avg       0.97      0.88      0.92       455
weighted avg       0.98      0.98      0.98       455



In [ ]:
cm = confusion_matrix(labels, preds)

print(cm)

[[384   0   2   0   0   0]
 [  0   9   0   0   0   0]
 [  4   0  27   0   0   0]
 [  2   0   0   6   0   0]
 [  0   0   2   0  14   0]
 [  0   0   1   0   0   4]]


## 輸出 embeddings

In [ ]:
model.eval()

embeddings = []

texts = all_user_df["text"].tolist()

batch_size = 32

In [ ]:
for i in range(0, len(texts), batch_size):

    batch_texts = texts[i:i+batch_size]

    inputs = tokenizer(
        batch_texts,
        padding=True,
        truncation=True,
        max_length=64,
        return_tensors="pt"
    )

    inputs = {
        k: v.to(model.device)
        for k, v in inputs.items()
    }

    with torch.no_grad():

        outputs = model.base_model(**inputs)

        cls_embedding = outputs.last_hidden_state[:,0,:]

    embeddings.append(
        cls_embedding.cpu().numpy()
    )

In [ ]:
embeddings = np.concatenate(embeddings, axis=0)

In [ ]:
print(embeddings.shape)

(2613, 768)


In [ ]:
np.save(
    "receipt_embeddings.npy",
    embeddings
)

# Benchmark

## BERT 但只有品名

In [ ]:
tokenizer_2 = AutoTokenizer.from_pretrained(model_name)

In [ ]:
train_encodings_2 = tokenizer_2(
    train_df["item_clean"].tolist(),
    truncation=True,
    padding=True,
    max_length=64
)


test_encodings_2 = tokenizer_2(
    test_df["item_clean"].tolist(),
    truncation=True,
    padding=True,
    max_length=64
)

In [ ]:
train_dataset_2 = ReceiptDataset(
    train_encodings_2,
    train_df["label_id"].tolist()
)


test_dataset_2 = ReceiptDataset(
    test_encodings_2,
    test_df["label_id"].tolist()
)

In [ ]:
model_2 = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id,
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: ckiplab/bert-base-chinese
Key                                        | Status     | 
-------------------------------------------+------------+-
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
bert.pooler.dense.bias                     | MISSING    | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 
bert.pooler.dense.weight                   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you ex

In [ ]:
training_args_2 = TrainingArguments(

    output_dir="./results_2",

    learning_rate=2e-5,

    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,

    num_train_epochs=5,

    weight_decay=0.01,

    eval_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,

    report_to="none"
)

In [ ]:
trainer_2 = Trainer(

    model=model_2,

    args=training_args_2,

    train_dataset=train_dataset_2,

    eval_dataset=test_dataset_2,

    compute_metrics=compute_metrics,
)

In [ ]:
trainer_2.train()

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1
1,No log,0.180750,0.953846,0.709563,0.948799
2,No log,0.173488,0.951648,0.819327,0.948520
3,No log,0.185273,0.953846,0.764638,0.951620
4,0.149870,0.179813,0.964835,0.871277,0.964218
5,0.149870,0.184433,0.962637,0.866355,0.962117


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=675, training_loss=0.12241244987205223, metrics={'train_runtime': 191.0377, 'train_samples_per_second': 56.481, 'train_steps_per_second': 3.533, 'total_flos': 354883780892160.0, 'train_loss': 0.12241244987205223, 'epoch': 5.0})

### Result

In [ ]:
pred_output_2 = trainer_2.predict(test_dataset_2)

In [ ]:
preds_2 = np.argmax(pred_output_2.predictions, axis=1)

labels = test_df["label_id"].values

In [ ]:
print(classification_report(
    labels,
    preds_2,
    target_names=list(label2id.keys())
))

              precision    recall  f1-score   support

          飲食       0.96      1.00      0.98       386
          交通       0.78      0.78      0.78         9
          購物       0.86      0.61      0.72        31
          娛樂       1.00      0.75      0.86         8
          教育       1.00      0.75      0.86        16
        醫療健康       0.67      0.80      0.73         5

    accuracy                           0.95       455
   macro avg       0.88      0.78      0.82       455
weighted avg       0.95      0.95      0.95       455



In [ ]:
cm_2 = confusion_matrix(labels, preds_2)

print(cm_2)

[[385   0   1   0   0   0]
 [  2   7   0   0   0   0]
 [ 10   0  19   0   0   2]
 [  2   0   0   6   0   0]
 [  1   1   2   0  12   0]
 [  0   1   0   0   0   4]]


## TF-IDF + Logistic Regression

In [ ]:
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.linear_model import LogisticRegression

from sklearn.neural_network import MLPClassifier

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

In [ ]:
vectorizer = TfidfVectorizer(

    max_features=10000,

    ngram_range=(1,2),

    min_df=2,

    sublinear_tf=True
)

In [ ]:
X_train = vectorizer.fit_transform(
    train_df["text"]
)

X_test = vectorizer.transform(
    test_df["text"]
)

In [ ]:
y_train = train_df["label_id"]

y_test = test_df["label_id"]

In [ ]:
lr_model = LogisticRegression(

    max_iter=2000,

    class_weight="balanced",

    random_state=42
)

In [ ]:
lr_model.fit(X_train, y_train)

LogisticRegression(class_weight='balanced', max_iter=2000, random_state=42)

### Result

In [ ]:
lr_preds = lr_model.predict(X_test)

In [ ]:
print(
    classification_report(
        y_test,
        lr_preds,
        target_names=list(label2id.keys())
    )
)

=== TF-IDF + Logistic Regression ===
              precision    recall  f1-score   support

          飲食       0.94      0.72      0.81       386
          交通       1.00      0.33      0.50         9
          購物       0.31      0.61      0.41        31
          娛樂       0.43      0.38      0.40         8
          教育       0.12      0.38      0.18        16
        醫療健康       0.05      0.40      0.09         5

    accuracy                           0.68       455
   macro avg       0.47      0.47      0.40       455
weighted avg       0.85      0.68      0.74       455



In [ ]:
lr_cm = confusion_matrix(
    y_test,
    lr_preds
)

print(lr_cm)

[[276   0  39   4  42  25]
 [  6   3   0   0   0   0]
 [  4   0  19   0   1   7]
 [  1   0   1   3   0   3]
 [  6   0   1   0   6   3]
 [  1   0   2   0   0   2]]


## TF-IDF + MLP

In [ ]:
mlp_model = MLPClassifier(

    hidden_layer_sizes=(256,128),

    activation="relu",

    solver="adam",

    batch_size=32,

    learning_rate_init=1e-3,

    max_iter=30,

    early_stopping=True,

    random_state=42
)

In [ ]:
mlp_model.fit(X_train, y_train)

MLPClassifier(batch_size=32, early_stopping=True, hidden_layer_sizes=(256, 128),
              max_iter=30, random_state=42)

### Result

In [ ]:
mlp_preds = mlp_model.predict(X_test)

In [ ]:
print(
    classification_report(
        y_test,
        mlp_preds,
        target_names=list(label2id.keys())
    )
)

=== TF-IDF + MLP ===
              precision    recall  f1-score   support

          飲食       0.90      0.89      0.89       386
          交通       1.00      0.33      0.50         9
          購物       0.33      0.48      0.39        31
          娛樂       1.00      0.38      0.55         8
          教育       0.26      0.38      0.31        16
        醫療健康       0.00      0.00      0.00         5

    accuracy                           0.81       455
   macro avg       0.58      0.41      0.44       455
weighted avg       0.83      0.81      0.81       455



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
mlp_cm = confusion_matrix(
    y_test,
    mlp_preds
)

print(mlp_cm)

[[342   0  28   0  16   0]
 [  6   3   0   0   0   0]
 [ 15   0  15   0   1   0]
 [  5   0   0   3   0   0]
 [ 10   0   0   0   6   0]
 [  3   0   2   0   0   0]]
